<a href="https://colab.research.google.com/github/QuynhAnh2101/Chi_tay/blob/main/Ch%E1%BB%89_tay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
DATASET_PATH = '/content/drive/MyDrive/Chỉ tayyyyy'
IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32
EPOCHS = 20
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)
print("--- Đang tải dữ liệu TRAIN (80%) ---")
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)
print("\n--- Đang tải dữ liệu VALIDATION (20%) ---")
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)
print("\nCác lớp nhận diện được tự động gán nhãn:")
print(train_generator.class_indices)
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("\n--- Bắt đầu Train mô hình ---")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)
model.save('/content/drive/MyDrive/mau_nhan_dien_chi_tay.keras')
print("\n Đã huấn luyện xong và lưu model tại MyDrive với tên 'mau_nhan_dien_chi_tay'")

--- Đang tải dữ liệu TRAIN (80%) ---
Found 162 images belonging to 2 classes.

--- Đang tải dữ liệu VALIDATION (20%) ---
Found 40 images belonging to 2 classes.

Các lớp nhận diện được tự động gán nhãn:
{'Có chữ M': 0, 'Tay bình thường': 1}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Bắt đầu Train mô hình ---
Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 46s 7s/step - accuracy: 0.5556 - loss: 1.0150 - val_accuracy: 0.5500 - val_loss: 0.6916
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 4s/step - accuracy: 0.5679 - loss: 0.6885 - val_accuracy: 0.4500 - val_loss: 0.7026
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.5062 - loss: 0.6962 - val_accuracy: 0.5500 - val_loss: 0.6861
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 0.5432 - loss: 0.6832 - val_accuracy: 0.5500 - val_loss: 0.6806
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - accuracy: 0.5679 - loss: 0.6888 - val_accuracy: 0.5500 - val_loss: 0.6836
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.5494 - loss: 0.6686 - val_accuracy: 0.5500 - val_loss: 0.7024
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.5556 - loss: 0.6961 - val_accuracy: 0.5500 - val_loss: 0.7214
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.6358 - loss: 0.6267 - val_accuracy: 0.5

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load model
model = load_model('/content/drive/MyDrive/mau_nhan_dien_chi_tay.keras')

def dự_đoán_chỉ_tay(đường_dẫn_ảnh):
    img = image.load_img(đường_dẫn_ảnh, target_size=(150, 150))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)[0][0]

    # Dựa vào train_generator.class_indices:
    # Thường 'Có chữ M' xếp trước nên giá trị gần 0 là Có chữ M, gần 1 là Tay bình thường
    if prediction < 0.5:
        phần_trăm = (1 - prediction) * 100
        print(f"Kết quả: CÓ CHỮ M ({phần_trăm:.2f}%)")
    else:
        phần_trăm = prediction * 100
        print(f"Kết quả: TAY BÌNH THƯỜNG ({phần_trăm:.2f}%)")

# Điền đường dẫn tới một file ảnh cụ thể để test thử
dự_đoán_chỉ_tay('/content/Screenshot 2026-05-30 203500.png')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
Kết quả: CÓ CHỮ M (94.16%)
